In [2]:
import pandas as pd
import numpy as np
import json

In [3]:
po_data = pd.read_json('C:\\Users\\N_R_KHAN\\Documents\\ITL_PROJECTS\\Proofreading_central_planning\\VS\\notebooks\\po_json\\response_1785297314383.json')
print(po_data.head())

                                           2112-09-09 10:23:39
po_number                                           4502859819
factory_id                                            36014990
date_of_mfr                                             2026/7
items        [{'item_code': '2001431649', 'item_description...


In [4]:
def product_code_matching(work_order_product_code, purchase_order_product_code):
    if pd.isna(work_order_product_code) or pd.isna(purchase_order_product_code):
        return False

    
    return str(work_order_product_code).strip() == str(purchase_order_product_code).strip()


wo_product_code = "LB 07655 C/1 / PPK-1023"
po_item_description = "LBL LB07655 PPK1023 C1 MWW001 416213/SPT / LB 07655 PPK-1023 C1"

print(product_code_matching(wo_product_code, po_item_description))  # Output: True


False


In [5]:
import re
from collections import Counter


def extract_code_parts(text: str) -> list[str]:
    """
    Converts:
        'LB 07655 C/1 / PPK-1023'
    into:
        ['LB', '07655', 'C', '1', 'PPK', '1023']

    Also converts:
        'LB07655 PPK1023 C1'
    into the same comparable parts.
    """
    return re.findall(r"[A-Z]+|\d+", (text or "").upper())


def product_code_matches(
    wo_product_code: str,
    po_item_description: str,
) -> bool:
    wo_parts = Counter(extract_code_parts(wo_product_code))
    po_parts = Counter(extract_code_parts(po_item_description))

    # Every WO component, including repeated components,
    # must be available in the PO description.
    return all(
        po_parts[part] >= required_count
        for part, required_count in wo_parts.items()
    )


wo_product_code = "LB 07655 C/1 / PPK-1023"
po_item_description = (
    "LBL LB07655 PPK1023 C1 MWW001 "
    "416213/SPT / LB 07655 PPK-1023 C1"
)

matched = product_code_matches(
    wo_product_code,
    po_item_description,
)

print(matched)  # True

True


In [6]:
tests = [
    (
        "ABC-458 / RED-12 c/1",
        "Product RED12 Size XL p1 ABC-458 c2",
    ),
    (
        "XYZ 1007 B/4",
        "Description: B4 XYZ-1007",
    ),
    (
        "JKL-900 A/2",
        "Description: JKL900 A2",
    ),
]

for wo_code, po_description in tests:
    print(product_code_matches(wo_code, po_description))



True
True
True


In [10]:
wo_product_code = "LB 07655 C/1 / PPK-1023"

parts = [
    p for p in wo_product_code.split()
    if p != "/"
]

all_parts = set()

for part in parts:
    all_parts.update({
        part,
        part.replace("/", ""),
        part.replace("-", ""),
        part.replace("/", "").replace("-", "")
    })

print(all_parts)

{'C1', '07655', 'PPK1023', 'C/1', 'LB', 'PPK-1023'}


In [14]:
wo_product_code = "LB 07655 C/1 / PPK-1023 2"
po_item_description = "LBL LB07655 PPK1023 C1 MWW001 416213/SPT"

# Build dictionary
parts = {}

for part in wo_product_code.upper().split():
    if part == "/":
        continue

    parts[part] = list(dict.fromkeys([
        part,
        part.replace("/", ""),
        part.replace("-", ""),
        part.replace("/", "").replace("-", ""),
    ]))

print(parts)

po_text = po_item_description.upper()

is_match = all(
    any(value in po_text for value in values)
    for values in parts.values()
)

print(is_match)

{'LB': ['LB'], '07655': ['07655'], 'C/1': ['C/1', 'C1'], 'PPK-1023': ['PPK-1023', 'PPK1023'], '2': ['2']}
True


In [ ]:
wo_product_code = "LB 07655 C/1 / PPK-1023 "
po_item_description = "LBL LB07655 PPK1023 C1 MWW001 416213/SPT / LB 07655 PPK-1023 C1"

def product_code_status(
    work_order_value: Any,
    purchase_order_value: Any,
) -> str:
    if is_missing(work_order_value) or is_missing(purchase_order_value):
        return "missing"

    wo_product_code = str(work_order_value).upper()
    po_item_description = str(purchase_order_value).upper()

    parts = {}

    for part in wo_product_code.split():
        if part == "/":
            continue

        parts[part] = list(
            dict.fromkeys(
                [
                    part,
                    part.replace("/", ""),
                    part.replace("-", ""),
                    part.replace("/", "").replace("-", ""),
                ]
            )
        )

    if not parts:
        return "missing"

    po_tokens = po_item_description.split()

    is_match = all(
        any(value in po_tokens for value in possible_values)
        for possible_values in parts.values()
    )

    return "match" if is_match else "mismatch"

{'LB': ['LB'], '07655': ['07655'], 'C/1': ['C/1', 'C1'], 'PPK-1023': ['PPK-1023', 'PPK1023']}
['LBL', 'LB07655', 'PPK1023', 'C1', 'MWW001', '416213/SPT', '/', 'LB', '07655', 'PPK-1023', 'C1']
True
